In [ ]:
!pip install open_clip_torch torch torchvision Pillow -q

In [ ]:
import os
import torch
import open_clip
from PIL import Image
from typing import List, Tuple, Dict, Any

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

# Породы деревьев

In [ ]:
# =============================================================================
# 1. ГИПЕРПАРАМЕТРЫ И НАСТРОЙКИ (Ваш код)
# =============================================================================

# Этот блок оставлен для контекста, но для оценки CLIP мы будем
# использовать "сырые" изображения, а не эти трансформации.
IMG_SIZE = 380
BATCH_SIZE = 16
RANDOM_SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Переименуем в DEVICE, чтобы соответствовать коду CLIP
DEVICE = device
print(f"Используется устройство: {DEVICE}")

# =============================================================================
# 2. ПОДГОТОВКА ДАННЫХ (Ваш код, с небольшими изменениями)
# =============================================================================

# --- Трансформации для EfficientNet (не используются в этой задаче) ---
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- Создание датасета ---
# Важно: мы НЕ будем применять трансформации к `val_dataset` сразу,
# так как нам нужны исходные PIL изображения для CLIP.
full_dataset = datasets.ImageFolder(root='/kaggle/input/tree-classification')
class_names_from_folders = full_dataset.classes
num_classes = len(class_names_from_folders)
print(f"Найдено {num_classes} классов в датасете: {class_names_from_folders}")

# --- Разделение на train/val ---
torch.manual_seed(RANDOM_SEED)
val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size
# Мы разделяем `full_dataset`, у которого пока нет трансформаций
train_subset, val_subset = random_split(full_dataset, [train_size, val_size])

# Теперь можно было бы назначить трансформации, если бы мы обучали модель:
# train_subset.dataset.transform = train_transforms
# val_subset.dataset.transform = val_transforms
# Но для оценки CLIP нам это не нужно.


# =============================================================================
# 3. ЛОГИКА МОДЕЛИ BioCLIP (Ваш код)
# =============================================================================

# !pip install open_clip_torch torch torchvision Pillow -q
MODELS = {}
cache_directory = "./models_cache"

SPECIES_LIST: List[Tuple[str, str]] = [
    # Новые классы из вашего датасета
    ("Боярышник однопестичный", "Crataegus monogyna"),
    ("Дёрен пестролистный", "Cornus alba"), # Упрощено для модели
    ("Кизильник блестящий", "Cotoneaster lucidus"),
    ("Спирея серая", "Spiraea cinerea"),
    ("Чубушник венечный", "Philadelphus coronarius"),

    # Старые классы
    ("берёза повислая", "Betula pendula"),
    ("вишня обыкновенная", "Prunus cerasus"),
    ("дуб красный", "Quercus rubra"),
    ("дуб черешчатый", "Quercus robur"),
    ("ель европейская", "Picea abies"),
    ("ива плакучая", "Salix babylonica"),
    ("каштан конский", "Aesculus hippocastanum"), # Исправлена опечатка hippocastingum -> hippocastanum
    ("клён платановидный", "Acer platanoides"),
    ("клён-явор", "Acer pseudoplanatus"),
    ("липа крупнолистная", "Tilia platyphyllos"),
    ("липа мелколистная", "Tilia cordata"),
    ("рябина обыкновенная", "Sorbus aucuparia"),
    ("сирень обыкновенная", "Syringa vulgaris"),
    ("тополь белый", "Populus alba"),
    ("черёмуха обыкновенная", "Prunus padus"),
    ("яблоня садовая", "Malus domestica"),
    ("ясень обыкновенный", "Fraxinus excelsior"),
]

def load_all_models():
    if MODELS:
        return
    print("Загрузка модели BioCLIP...")
    model_bio, _, preprocess_bio = open_clip.create_model_and_transforms(
        "hf-hub:imageomics/bioclip", cache_dir=cache_directory)
    MODELS['bioclip_model'] = model_bio.to(DEVICE).eval()
    MODELS['bioclip_preprocess'] = preprocess_bio
    MODELS['bioclip_tokenizer'] = open_clip.get_tokenizer("hf-hub:imageomics/bioclip")
    print("Модель загружена.")

def build_species_prompts(latin_name: str) -> List[str]:
    return [
        f"a photo of a {latin_name}", f"a {latin_name} tree", f"leaves of a {latin_name}",
        f"bark of a {latin_name}", f"a branch of {latin_name}",
    ]

def init_bioclip_species_cache(models: Dict[str, Any], species_list: List[Tuple[str, str]]) -> None:
    print("Инициализация кеша текстовых промптов...")
    all_prompts, prompt_map = [], []
    for i, (_, latin_name) in enumerate(species_list):
        ps = build_species_prompts(latin_name)
        all_prompts.extend(ps)
        prompt_map.extend([i] * len(ps))
    text_tokens = models['bioclip_tokenizer'](all_prompts).to(DEVICE)
    with torch.no_grad():
        text_features = models['bioclip_model'].encode_text(text_tokens)
        text_features /= text_features.norm(dim=-1, keepdim=True)
    MODELS['bioclip_cache'] = {
        "species_list": species_list, "prompt_map": prompt_map, "text_features": text_features,
    }
    print("Кеш инициализирован.")

def get_species_with_bioclip(object_image: Image.Image, top_k: int = 3) -> List[Tuple[str, float]]:
    load_all_models()
    cache = MODELS.get('bioclip_cache')
    if cache is None:
        init_bioclip_species_cache(MODELS, SPECIES_LIST)
        cache = MODELS['bioclip_cache']

    species_list, prompt_map, text_features = cache["species_list"], cache["prompt_map"], cache["text_features"]
    if object_image.mode != "RGB":
        object_image = object_image.convert("RGB")
    image_input = MODELS['bioclip_preprocess'](object_image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        image_features = MODELS['bioclip_model'].encode_image(image_input)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        all_logits = (100.0 * image_features @ text_features.T).squeeze(0)

    num_species = len(species_list)
    species_logits = torch.full((num_species,), -float('inf'), device=all_logits.device)
    for prompt_idx, logit in enumerate(all_logits):
        sp_idx = prompt_map[prompt_idx]
        species_logits[sp_idx] = torch.max(species_logits[sp_idx], logit)

    probs = species_logits.softmax(dim=-1)
    k = min(top_k, num_species)
    top_probs, top_idx = torch.topk(probs, k=k)
    results = [(species_list[int(idx)][0], round(float(p), 4)) for p, idx in zip(top_probs, top_idx)]
    return results


# =============================================================================
# 4. ОЦЕНКА КАЧЕСТВА BioCLIP
# =============================================================================

if __name__ == '__main__':
    print("\n--- НАЧАЛО ОЦЕНКИ МОДЕЛИ BioCLIP ---")

    # 1. Загружаем модель и кешируем текстовые признаки
    load_all_models()
    init_bioclip_species_cache(MODELS, SPECIES_LIST)

    # 2. Создаем сопоставление между классами CLIP и классами датасета
    # Это самый важный шаг для корректного сравнения!
    dataset_class_to_idx = full_dataset.class_to_idx
    
    # Создаем словарь: {латинское_название_из_папки: индекс_класса}
    # Например: {'Betula pendula': 0, 'Tilia cordata': 1, ...}
    print("\nСопоставление классов из датасета (папок):")
    print(dataset_class_to_idx)
    
    clip_rus_name_to_dataset_idx = {}
    clip_class_names_ordered = [s[0] for s in SPECIES_LIST]
    
    for rus_name, latin_name in SPECIES_LIST:
        # Правильно: ищем русское имя (rus_name) среди ключей датасета,
        # которые тоже являются русскими именами.
        idx = dataset_class_to_idx.get(rus_name) 
        if idx is not None:
            clip_rus_name_to_dataset_idx[rus_name] = idx
        else:
            # Это поможет вам увидеть, какие классы из SPECIES_LIST не нашлись в датасете
            print(f"Внимание: класс '{rus_name}' из SPECIES_LIST не найден в папках датасета.")

    print("\nСопоставление русских названий CLIP с индексами классов датасета:")
    print(clip_rus_name_to_dataset_idx)
    
    # 3. Проходим по валидационному датасету и делаем предсказания
    all_true_labels = []
    all_pred_labels = []

    # Мы будем итерироваться по `val_subset`, который содержит индексы
    # и позволяет получить исходные данные (картинку и метку)
    for i in tqdm(range(len(val_subset)), desc="Оценка на валидационном наборе"):
        # Получаем PIL изображение и его настоящий индекс класса
        pil_image, true_label_idx = val_subset[i]

        # Делаем предсказание с помощью CLIP
        # Нам нужен только топ-1 результат для классификации
        predictions = get_species_with_bioclip(pil_image, top_k=1)

        if predictions:
            # Получаем русское название предсказанного класса
            predicted_rus_name = predictions[0][0]
            # Находим индекс этого класса в нашем датасете
            predicted_label_idx = clip_rus_name_to_dataset_idx.get(predicted_rus_name, -1) # -1, если класс не найден
        else:
            predicted_label_idx = -1 # Если модель ничего не вернула

        all_true_labels.append(true_label_idx)
        all_pred_labels.append(predicted_label_idx)

    # 4. Считаем и выводим метрики
    
    # Получим имена классов в правильном порядке для отчета
    # Создаем список имен, где индекс элемента соответствует метке класса
    report_class_names = ["" for _ in range(num_classes)]
    for class_name, idx in dataset_class_to_idx.items():
        report_class_names[idx] = class_name
        
    print("\n--- РЕЗУЛЬТАТЫ ОЦЕНКИ ---")
    
    # Общая точность (Accuracy)
    accuracy = accuracy_score(all_true_labels, all_pred_labels)
    print(f"Общая точность (Accuracy): {accuracy:.2%}\n")

    # Подробный отчет (precision, recall, f1-score)
    report = classification_report(
        all_true_labels,
        all_pred_labels,
        target_names=report_class_names,
        zero_division=0
    )
    print("Детальный отчет по классам:")
    print(report)

    # =============================================================================
    # 5. ВИЗУАЛИЗАЦИЯ МАТРИЦЫ ОШИБОК
    # =============================================================================

    print("\nПостроение матрицы ошибок...")

    # 1. Вычисляем матрицу ошибок
    cm = confusion_matrix(all_true_labels, all_pred_labels)

    # 2. Создаем фигуру для графика
    # Размер можно подобрать в зависимости от количества классов
    plt.figure(figsize=(14, 12))

    # 3. Используем seaborn для красивой визуализации
    sns.heatmap(
        cm,
        annot=True,        # Показать числа в ячейках
        fmt='d',           # Форматировать числа как целые
        cmap='Blues',      # Цветовая схема
        xticklabels=report_class_names, # Метки по оси X
        yticklabels=report_class_names  # Метки по оси Y
    )

    # 4. Добавляем названия и поворачиваем метки для читаемости
    plt.title('Матрица ошибок (Confusion Matrix)', fontsize=16)
    plt.ylabel('Истинные классы (True Class)', fontsize=12)
    plt.xlabel('Предсказанные классы (Predicted Class)', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout() # Автоматически подбирает расположение элементов

    # 5. Показываем график
    plt.show()

# Болезни

In [ ]:
import os
import torch
import open_clip
from PIL import Image
from typing import List, Tuple, Dict, Any

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# =============================================================================
# 1. ГИПЕРПАРАМЕТРЫ И НАСТРОЙКИ
# =============================================================================

# !!! ВАЖНО: УКАЖИТЕ ПУТЬ К ВАШЕМУ ДАТАСЕТУ С БОЛЕЗНЯМИ !!!
DATA_ROOT = '/kaggle/input/diseases-dataset' 
# Например: '/kaggle/input/tree-diseases-dataset'

RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используется устройство: {DEVICE}")

# =============================================================================
# 2. ПОДГОТОВКА ДАННЫХ
# =============================================================================

# Создаем датасет. Убедитесь, что названия папок в DATA_ROOT 
# в точности совпадают с русскими названиями из DISEASE_LIST ниже.
try:
    full_dataset = datasets.ImageFolder(root=DATA_ROOT)
    class_names_from_folders = full_dataset.classes
    num_classes = len(class_names_from_folders)
    print(f"Найдено {num_classes} классов болезней в датасете: {class_names_from_folders}")
except FileNotFoundError:
    print(f"ОШИБКА: Директория не найдена по пути '{DATA_ROOT}'.")
    print("Пожалуйста, измените переменную DATA_ROOT в коде на правильный путь.")
    exit() # Прерываем выполнение, если датасет не найден

# Разделение на train/val
torch.manual_seed(RANDOM_SEED)
val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_subset, val_subset = random_split(full_dataset, [train_size, val_size])

# =============================================================================
# 3. ЛОГИКА МОДЕЛИ BioCLIP ДЛЯ БОЛЕЗНЕЙ
# =============================================================================

MODELS = {}
cache_directory = "./models_cache"

# --- НОВЫЙ СПИСОК КЛАССОВ: БОЛЕЗНИ ---
# Формат: (Русское название для сопоставления с папкой, Английское/описательное название для промпта)
DISEASE_LIST: List[Tuple[str, str]] = [
    ("стволовая гниль", "stem rot on a tree"),
    ("комплексное усыхание", "complex drying of a tree"),
    ("голландская болезнь вяза", "dutch elm disease"),
    ("естественное отмирание ветвей", "natural branch dieback"),
    ("березовый трутовик", "birch polypore fungus"),
    ("вертициллезное увядание", "verticillium wilt"),
    ("ядровая гниль, вызванная дереворазрушающими грибами", "heart rot caused by wood-destroying fungi"),
    ("корневая или комлевая гниль", "root or butt rot"),
    ("механическое повреждение ствола", "mechanical damage to the trunk"),
    ("халаровый некроз", "chalara dieback of ash"),
    ("повреждение насекомыми-филлофагами", "damage by phyllophagous insects"),
    ("ржавчина", "rust disease on leaves"),
    ("фитофтороз", "phytophthora root rot"),
    ("рак", "canker on a tree branch"),
    ("корневая губка", "root sponge fungus"),
    ("мучнистая роса", "powdery mildew on leaves"),
]

def load_all_models():
    if MODELS: return
    print("Загрузка модели BioCLIP...")
    model_bio, _, preprocess_bio = open_clip.create_model_and_transforms(
        "hf-hub:imageomics/bioclip", cache_dir=cache_directory)
    MODELS['bioclip_model'] = model_bio.to(DEVICE).eval()
    MODELS['bioclip_preprocess'] = preprocess_bio
    MODELS['bioclip_tokenizer'] = open_clip.get_tokenizer("hf-hub:imageomics/bioclip")
    print("Модель загружена.")

# --- НОВАЯ ФУНКЦИЯ ДЛЯ ГЕНЕРАЦИИ ПРОМПТОВ О БОЛЕЗНЯХ ---
def build_disease_prompts(english_desc: str) -> List[str]:
    return [
        f"a photo of a tree with {english_desc}",
        f"signs of {english_desc} on a tree trunk",
        f"{english_desc} affecting tree leaves",
        f"a close-up of {english_desc}",
        f"a diseased tree, showing symptoms of {english_desc}",
    ]

def init_bioclip_disease_cache(models: Dict[str, Any], disease_list: List[Tuple[str, str]]) -> None:
    print("Инициализация кеша текстовых промптов для болезней...")
    all_prompts, prompt_map = [], []
    for i, (_, english_desc) in enumerate(disease_list):
        ps = build_disease_prompts(english_desc) # Используем новую функцию
        all_prompts.extend(ps)
        prompt_map.extend([i] * len(ps))
    text_tokens = models['bioclip_tokenizer'](all_prompts).to(DEVICE)
    with torch.no_grad():
        text_features = models['bioclip_model'].encode_text(text_tokens)
        text_features /= text_features.norm(dim=-1, keepdim=True)
    MODELS['bioclip_cache'] = {
        "disease_list": disease_list, "prompt_map": prompt_map, "text_features": text_features,
    }
    print("Кеш инициализирован.")

def get_disease_with_bioclip(object_image: Image.Image, top_k: int = 1) -> List[Tuple[str, float]]:
    load_all_models()
    cache = MODELS.get('bioclip_cache')
    if cache is None:
        init_bioclip_disease_cache(MODELS, DISEASE_LIST)
        cache = MODELS['bioclip_cache']

    disease_list, prompt_map, text_features = cache["disease_list"], cache["prompt_map"], cache["text_features"]
    if object_image.mode != "RGB":
        object_image = object_image.convert("RGB")
    image_input = MODELS['bioclip_preprocess'](object_image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        image_features = MODELS['bioclip_model'].encode_image(image_input)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        all_logits = (100.0 * image_features @ text_features.T).squeeze(0)

    num_diseases = len(disease_list)
    disease_logits = torch.full((num_diseases,), -float('inf'), device=all_logits.device)
    for prompt_idx, logit in enumerate(all_logits):
        sp_idx = prompt_map[prompt_idx]
        disease_logits[sp_idx] = torch.max(disease_logits[sp_idx], logit)

    probs = disease_logits.softmax(dim=-1)
    k = min(top_k, num_diseases)
    top_probs, top_idx = torch.topk(probs, k=k)
    results = [(disease_list[int(idx)][0], round(float(p), 4)) for p, idx in zip(top_probs, top_idx)]
    return results

# =============================================================================
# 4. ОЦЕНКА КАЧЕСТВА BioCLIP
# =============================================================================

if __name__ == '__main__':
    print("\n--- НАЧАЛО ОЦЕНКИ МОДЕЛИ BioCLIP НА БОЛЕЗНЯХ ---")

    load_all_models()
    init_bioclip_disease_cache(MODELS, DISEASE_LIST)

    dataset_class_to_idx = full_dataset.class_to_idx
    print("\nСопоставление классов из датасета (папок):")
    print(dataset_class_to_idx)
    
    clip_rus_name_to_dataset_idx = {}
    for rus_name, _ in DISEASE_LIST:
        idx = dataset_class_to_idx.get(rus_name) 
        if idx is not None:
            clip_rus_name_to_dataset_idx[rus_name] = idx
        else:
            print(f"Внимание: класс '{rus_name}' из DISEASE_LIST не найден в папках датасета.")

    print("\nСопоставление русских названий CLIP с индексами классов датасета:")
    print(clip_rus_name_to_dataset_idx)
    
    all_true_labels, all_pred_labels = [], []

    for i in tqdm(range(len(val_subset)), desc="Оценка на валидационном наборе"):
        pil_image, true_label_idx = val_subset[i]
        predictions = get_disease_with_bioclip(pil_image, top_k=1)
        
        predicted_label_idx = -1 # Значение по умолчанию, если предсказание не удалось
        if predictions:
            predicted_rus_name = predictions[0][0]
            predicted_label_idx = clip_rus_name_to_dataset_idx.get(predicted_rus_name, -1)

        all_true_labels.append(true_label_idx)
        all_pred_labels.append(predicted_label_idx)

    # --- РАСЧЕТ И ВЫВОД МЕТРИК ---
    report_class_names = ["" for _ in range(num_classes)]
    for class_name, idx in dataset_class_to_idx.items():
        report_class_names[idx] = class_name
        
    print("\n--- РЕЗУЛЬТАТЫ ОЦЕНКИ ---")
    
    accuracy = accuracy_score(all_true_labels, all_pred_labels)
    print(f"Общая точность (Accuracy): {accuracy:.2%}\n")

    report = classification_report(
        all_true_labels, all_pred_labels,
        target_names=report_class_names, zero_division=0
    )
    print("Детальный отчет по классам:")
    print(report)

    # =============================================================================
    # 5. ВИЗУАЛИЗАЦИЯ МАТРИЦЫ ОШИБОК
    # =============================================================================
    print("\nПостроение матрицы ошибок...")
    cm = confusion_matrix(all_true_labels, all_pred_labels)
    plt.figure(figsize=(16, 14))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=report_class_names, yticklabels=report_class_names
    )
    plt.title('Матрица ошибок (Confusion Matrix)', fontsize=16)
    plt.ylabel('Истинные классы (True Class)', fontsize=12)
    plt.xlabel('Предсказанные классы (Predicted Class)', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()